In [ ]:
from git import Repo
import os
import json
import pandas as pd

In [ ]:
import os
import pandas as pd
import json
import mysql.connector


In [ ]:
# ---------- CONFIG ---------- #
path = "D:/guvi/project/phonepay/data/data/aggregated/transaction/country/india/state/"

mysql_config = {
    "host": "localhost",
    "user": "root",
    "password": "Amitpm@52",
    "database": "phonepe_insights"
}

In [ ]:
# ---my first data frame------- aggregated state list---------- #
import os
import json
import mysql.connector
import pandas as pd

# Path where your JSON files are stored
path = r"D:\guvi\project\phonepay\data\data\aggregated\transaction\country\india\state"

# MySQL connection config
mysql_config = {
    'host': 'localhost',
    'user': 'root',
    'password': "Amitpm@52",  # Update with your actual password
    'database': 'phonepe_insights'
}

# --- Aggregated State List ---
Agg_state_list = os.listdir(path)
print('Agg_state_list :', Agg_state_list)

clm = {
    'State': [],
    'Year': [],
    'Quarter': [],
    'Transaction_type': [],
    'Transaction_count': [],
    'Transaction_amount': []
}

# Loop through the state directories and JSON files
for i in Agg_state_list:
    p_i = os.path.join(path, i)
    Agg_yr = os.listdir(p_i)

    for j in Agg_yr:
        p_j = os.path.join(p_i, j)
        Agg_yr_list = os.listdir(p_j)

        for k in Agg_yr_list:
            if k.endswith('.json'):  # Process only JSON files
                p_k = os.path.join(p_j, k)
                with open(p_k, 'r', encoding='utf-8') as data_file:
                    D = json.load(data_file)
                    print(f"Loaded JSON: {p_k}")  # Debug: Print file path

                    # Check if the JSON contains 'transactionData'
                    if D.get('data') and D['data'].get('transactionData'):
                        print(D['data']['transactionData'])  # Debug: Print transaction data
                        for z in D['data']['transactionData']:
                            Name = z['name']
                            count = z['paymentInstruments'][0]['count']
                            amount = z['paymentInstruments'][0]['amount']

                            # Append to the column dictionary
                            clm['Transaction_type'].append(Name)
                            clm['Transaction_count'].append(count)
                            clm['Transaction_amount'].append(amount)
                            clm['State'].append(i)
                            clm['Year'].append(int(j))  # Convert year to integer
                            clm['Quarter'].append(int(k.strip('.json')))  # Convert quarter to integer

# Create DataFrame from the columns
Agg_Trans = pd.DataFrame(clm)
print("\nSample Aggregated Transaction DataFrame:")
print(Agg_Trans.head())

# Check if DataFrame is empty before proceeding
if Agg_Trans.empty:
    print("No data to insert!")
else:
    print(Agg_Trans.info())  # Debug: Check the info of the DataFrame

# ---------- Insert into MySQL ----------
try:
    conn = mysql.connector.connect(**mysql_config)
    cursor = conn.cursor()

    insert_sql = """
        INSERT INTO aggregated_transactions
        (State, Year, Quarter, Transaction_type, Transaction_count, Transaction_amount)
        VALUES (%s, %s, %s, %s, %s, %s)
    """

    # Insert each row into the database
    for index, row in Agg_Trans.iterrows():
        # Validation: Ensure that data is valid before insertion
        if row['Transaction_type'] and row['Transaction_count'] and row['Transaction_amount']:
            values = (
                row['State'],
                row['Year'],
                row['Quarter'],
                row['Transaction_type'],
                int(row['Transaction_count']),    # Safely cast to int
                float(row['Transaction_amount'])  # Safely cast to float
            )
            try:
                cursor.execute(insert_sql, values)
            except mysql.connector.Error as err:
                print(f"Error inserting record {values}: {err}")
        else:
            print(f"Skipping invalid row: {row}")  # Debug: Invalid rows will be printed

    conn.commit()
    print("Data inserted into MySQL successfully!")

except mysql.connector.Error as err:
    print("MySQL Error:", err)

finally:
    if conn.is_connected():
        cursor.close()
        conn.close()
        print("MySQL connection closed.")


In [ ]:
Agg_Trans

In [ ]:
# my second data frame  ..... user_data

# MySQL Config
conn = mysql.connector.connect(
    host="localhost",
    user="root",
    password="Amitpm@52",
    database="phonepe_insights"
)
cursor = conn.cursor()

# JSON directory
user_path = "D:/guvi/project/phonepay/data/data/aggregated/user/country/india/state/"

user_data = {
    'State': [], 'Year': [], 'Quarter': [], 'RegisteredUsers': [], 'AppOpens': []
}

states = os.listdir(user_path)

for state in states:
    state_path = os.path.join(user_path, state)
    years = os.listdir(state_path)

    for year in years:
        year_path = os.path.join(state_path, year)
        files = os.listdir(year_path)

        for file in files:
            file_path = os.path.join(year_path, file)
            with open(file_path, 'r') as f:
                data = json.load(f)

                if data['data']:
                    user_data['State'].append(state)
                    user_data['Year'].append(int(year))
                    user_data['Quarter'].append(int(file.strip('.json')))
                    user_data['RegisteredUsers'].append(data['data'].get('registeredUsers', 0))
                    user_data['AppOpens'].append(data['data'].get('appOpens', 0))

df_user = pd.DataFrame(user_data)
print(df_user)

In [ ]:

#my third data frame aggregated insurance data

# MySQL connection
conn = mysql.connector.connect(
    host="localhost",
    user="root",
    password="Amitpm@52",
    database="phonepe_insights"
)
cursor = conn.cursor()



# PATH #
insurance_path = "D:/guvi/project/phonepay/data/data/aggregated/insurance/country/india/state"


insurance_data = {
    'State': [], 'Year': [], 'Quarter': [],
    'Transaction_type': [], 'Transaction_count': [], 'Transaction_amount': []
}

# JSON FILES 
states = os.listdir(insurance_path)
for state in states:
    state_path = os.path.join(insurance_path, state)
    if not os.path.isdir(state_path):
        continue

    years = os.listdir(state_path)
    for year in years:
        year_path = os.path.join(state_path, year)
        if not os.path.isdir(year_path):
            continue

        files = os.listdir(year_path)
        for file in files:
            if not file.endswith(".json"):
                continue

            file_path = os.path.join(year_path, file)
            with open(file_path, 'r') as f:
                data = json.load(f)

                if data.get("data") and "transactionData" in data["data"]:
                    for txn in data["data"]["transactionData"]:
                        if txn["name"].lower() == "insurance":
                            insurance_data['State'].append(state)
                            insurance_data['Year'].append(int(year))
                            insurance_data['Quarter'].append(int(file.strip('.json')))
                            insurance_data['Transaction_type'].append(txn["name"])
                            insurance_data['Transaction_count'].append(txn['paymentInstruments'][0]['count'])
                            insurance_data['Transaction_amount'].append(txn['paymentInstruments'][0]['amount'])

# insurance dataframe
df_insurance = pd.DataFrame(insurance_data)
print("Filtered Insurance Data:")
print(df_insurance.head())

# -------- INSERT INTO MYSQL -------- #
if not df_insurance.empty:
    conn = mysql.connector.connect(
        host="localhost",
        user="root",
        password="Amitpm@52",
        database="phonepe_insights"
    )
    cursor = conn.cursor()

    for _, row in df_insurance.iterrows():
        sql = """
        INSERT INTO aggregated_insurance
        (State, Year, Quarter, Insurance_type, Insurance_count, Insurance_amount)
        VALUES (%s, %s, %s, %s, %s, %s)
        """
        values = (
            row['State'], row['Year'], row['Quarter'],
            row['Transaction_type'], row['Transaction_count'], row['Transaction_amount']
        )
        cursor.execute(sql, values)

    conn.commit()
    cursor.close()
    conn.close()
    print(" Insurance data inserted successfully!")
else:
    print(" No Insurance records found to insert.")


In [ ]:
#my fourth data frame   .....map 
#import os
import json
import pandas as pd
import mysql.connector

# Path to the transaction data
map_trans_path = "D:/guvi/project/phonepay/data/data/map/transaction/hover/country/india/state"

# Initialize the dictionary to store data
map_transaction_data = {
    'State': [], 'Year': [], 'Quarter': [], 
    'District': [], 'Transaction_count': [], 'Transaction_amount': []
}

# Loop through the files to read and extract data
for state in os.listdir(map_trans_path):
    state_path = os.path.join(map_trans_path, state)
    for year in os.listdir(state_path):
        year_path = os.path.join(state_path, year)
        for file in os.listdir(year_path):
            file_path = os.path.join(year_path, file)
            with open(file_path, 'r') as f:
                data = json.load(f)
                if data["data"] and "hoverDataList" in data["data"]:
                    for entry in data["data"]["hoverDataList"]:
                        district = entry["name"]
                        transaction_info = entry["metric"][0]

                        map_transaction_data["State"].append(state)
                        map_transaction_data["Year"].append(int(year))
                        map_transaction_data["Quarter"].append(int(file.strip('.json')))
                        map_transaction_data["District"].append(district)
                        map_transaction_data["Transaction_count"].append(transaction_info["count"])
                        map_transaction_data["Transaction_amount"].append(transaction_info["amount"])

# Create DataFrame
df_map_transaction = pd.DataFrame(map_transaction_data)

# MySQL database configuration
mysql_config = {
    "host": "localhost",
    "user": "root",
    "password": "Amitpm@52",  # Your MySQL password
    "database": "phonepe_insights"
}

# Insert data into MySQL database
try:
    conn = mysql.connector.connect(**mysql_config)
    cursor = conn.cursor()

    # SQL query to insert data
    insert_query = """
    INSERT INTO map_transaction (State, Year, Quarter, District, Transaction_count, Transaction_amount)
    VALUES (%s, %s, %s, %s, %s, %s)
    """

    # Loop through DataFrame rows and insert data into MySQL
    for index, row in df_map_transaction.iterrows():
        cursor.execute(insert_query, (
            row['State'], 
            row['Year'], 
            row['Quarter'], 
            row['District'], 
            row['Transaction_count'], 
            row['Transaction_amount']
        ))

    # Commit transaction
    conn.commit()
    print("Data inserted into map_transaction table successfully!")

except mysql.connector.Error as err:
    print("Error inserting data:", err)

finally:
    if conn.is_connected():
        cursor.close()
        conn.close()
        print("MySQL connection closed.")

In [ ]:
df_map_transaction.head()

In [ ]:
#my fifth data frame ...map_insurance

import os
import json
import pandas as pd
import mysql.connector

# ---------- PATH & DB CONFIG ---------- #
map_insurance_path = r"D:\guvi\project\phonepay\data\data\map\insurance\hover\country\india\state"
mysql_config = {
    "host": "localhost",
    "user": "root",
    "password": "Amitpm@52",   # Replace with your password
    "database": "phonepe_insights"
}

# ---------- Extract Data ---------- #
insurance_data = {
    'State': [], 'Year': [], 'Quarter': [],
    'District': [], 'Insurance_count': [], 'Insurance_amount': []
}

states = os.listdir(map_insurance_path)
for state in states:
    state_path = os.path.join(map_insurance_path, state)
    years = os.listdir(state_path)

    for year in years:
        year_path = os.path.join(state_path, year)
        files = os.listdir(year_path)

        for file in files:
            file_path = os.path.join(year_path, file)
            with open(file_path, 'r', encoding='utf-8') as f:
                data = json.load(f)

                if data.get('data') and data['data'].get('hoverDataList'):
                    for district_data in data['data']['hoverDataList']:
                        name = district_data['name']
                        metrics = district_data['metric'][0]

                        insurance_data['State'].append(state)
                        insurance_data['Year'].append(int(year))
                        insurance_data['Quarter'].append(int(file.strip('.json')))
                        insurance_data['District'].append(name)
                        insurance_data['Insurance_count'].append(metrics['count'])
                        insurance_data['Insurance_amount'].append(metrics['amount'])

# ---------- Convert to DataFrame ---------- #
df_map_insurance = pd.DataFrame(insurance_data)
print("Sample Extracted Data:\n", df_map_insurance.head())

# ---------- Insert into MySQL ---------- #
try:
    conn = mysql.connector.connect(**mysql_config)
    cursor = conn.cursor()

    for _, row in df_map_insurance.iterrows():
        sql = """
        INSERT INTO map_insurance
        (State, Year, Quarter, District, Insurance_count, Insurance_amount)
        VALUES (%s, %s, %s, %s, %s, %s)
        """
        values = (
            row['State'], row['Year'], row['Quarter'],
            row['District'], row['Insurance_count'], row['Insurance_amount']
        )
        cursor.execute(sql, values)

    conn.commit()
    print(" Map Insurance data inserted successfully!")

except mysql.connector.Error as err:
    print(" MySQL Error:", err)

finally:
    if conn.is_connected():
        cursor.close()
        conn.close()
        print(" MySQL connection closed.")
        
        

In [ ]:
#my sixth data frame.....map_user data extraction 
        

# Set your path
map_user_path = r"D:\guvi\project\phonepay\data\data\map\user\hover\country\india\state"

# Data container
map_user_data = {
    "State": [],
    "Year": [],
    "Quarter": [],
    "District": [],
    "RegisteredUsers": [],
    "AppOpens": []
}

# Traverse folders
for state in os.listdir(map_user_path):
    state_path = os.path.join(map_user_path, state)
    for year in os.listdir(state_path):
        year_path = os.path.join(state_path, year)
        for file in os.listdir(year_path):
            if file.endswith(".json"):
                quarter = int(file.strip(".json"))
                file_path = os.path.join(year_path, file)
                with open(file_path, "r", encoding="utf-8") as f:
                    data = json.load(f)
                    if data.get("data") and data["data"].get("hoverData"):
                        for district, details in data["data"]["hoverData"].items():
                            map_user_data["State"].append(state)
                            map_user_data["Year"].append(int(year))
                            map_user_data["Quarter"].append(quarter)
                            map_user_data["District"].append(district)
                            map_user_data["RegisteredUsers"].append(details.get("registeredUsers", 0))
                            map_user_data["AppOpens"].append(details.get("appOpens", 0))

# Convert to DataFrame
df_map_user = pd.DataFrame(map_user_data)
print("map_user:\n", df_map_user.head())

# MySQL Insert
try:
    conn = mysql.connector.connect(
        host="localhost",
        user="root",
        password="Amitpm@52",
        database="phonepe_insights"
    )
    cursor = conn.cursor()

    insert_query = """
        INSERT INTO map_user (State, Year, Quarter, District, RegisteredUsers, AppOpens)
        VALUES (%s, %s, %s, %s, %s, %s)
    """

    for _, row in df_map_user.iterrows():
        values = (
            row["State"], row["Year"], row["Quarter"],
            row["District"], row["RegisteredUsers"], row["AppOpens"]
        )
        cursor.execute(insert_query, values)

    conn.commit()
    print("Data inserted into `map_user` successfully!")

except mysql.connector.Error as err:
    print(" MySQL Error:", err)

finally:
    if conn.is_connected():
        cursor.close()
        conn.close()
        print("MySQL connection closed.")

In [ ]:
#my seventh data frame......top_user
import os
import json
import pandas as pd
import mysql.connector

# --- MySQL connection setup ---
conn = mysql.connector.connect(
    host="localhost",
    user="root",
    password="Amitpm@52",  # Replace with your password if different
    database="phonepe_insights"
)
cursor = conn.cursor()

# --- Data container: storing fields for top_user ---
top_user_data = {
    'State': [],
    'Pincode': [],
    'RegisteredUsers': [],
    'Year': [],
    'Quarter': []
}

# --- Define the base path where JSON files are stored ---
base_path = r"D:\guvi\project\phonepay\data\data\top\user\country\india\state"

# --- Walk through all JSON files in the directory structure ---
for root, dirs, files in os.walk(base_path):
    # Try to extract Year from folder name
    path_parts = root.split(os.sep)
    year = None
    if len(path_parts) >= 1 and path_parts[-1].isdigit():
        try:
            year = int(path_parts[-1])
        except ValueError:
            year = None
    
    # Process each file in the current directory
    for file in files:
        if file.endswith(".json"):
            try:
                quarter = int(file.split(".")[0])
            except ValueError:
                quarter = None
            file_path = os.path.join(root, file)
            try:
                with open(file_path, "r", encoding="utf-8") as f:
                    content = f.read()
                    content = content.replace("//Ignore. For internal use only.", "")
                    content = content.replace("...", "")
                    data = json.loads(content)
            except Exception as e:
                print(f"Error loading JSON from {file_path}: {e}")
                continue

            if not data or "data" not in data:
                print(f"Skipping file without valid 'data': {file_path}")
                continue

            data_section = data["data"]

            # Process state-level data
            states_list = data_section.get("states")
            if isinstance(states_list, list):
                for entry in states_list:
                    top_user_data['State'].append(entry.get("name"))
                    top_user_data['Pincode'].append(None)
                    top_user_data['RegisteredUsers'].append(entry.get("registeredUsers"))
                    top_user_data['Year'].append(year)
                    top_user_data['Quarter'].append(quarter)

            # Process pincode-level data
            pincodes_list = data_section.get("pincodes")
            if isinstance(pincodes_list, list):
                for entry in pincodes_list:
                    top_user_data['State'].append(None)
                    top_user_data['Pincode'].append(entry.get("name"))
                    top_user_data['RegisteredUsers'].append(entry.get("registeredUsers"))
                    top_user_data['Year'].append(year)
                    top_user_data['Quarter'].append(quarter)

# --- Create a DataFrame from the collected data ---
df_top_user = pd.DataFrame(top_user_data)
print("df_top_user:",df_top_user)
print(df_top_user.head())

# --- Insert the data from DataFrame into MySQL ---
insert_sql = """
INSERT INTO top_user (State, Pincode, RegisteredUsers, Year, Quarter)
VALUES (%s, %s, %s, %s, %s)
"""
for idx, row in df_top_user.iterrows():
    cursor.execute(insert_sql, (
        row['State'], row['Pincode'],
        row['RegisteredUsers'], row['Year'], row['Quarter']
    ))

conn.commit()
cursor.close()
conn.close()
print("Data inserted successfully into top_user!")

In [ ]:
#my eigth data frame .....top_transaction


import os
import json
import pandas as pd

# Path for top_transaction JSON files
top_trans_path = "D:/guvi/project/phonepay/data/data/top/transaction/country/india/state"

# Define a dictionary to collect data
top_transaction_data = {
    'State': [], 'Year': [], 'Quarter': [],
    'Pincode': [], 'Transaction_count': [], 'Transaction_amount': []
}

# Loop through the files
for state in os.listdir(top_trans_path):
    state_path = os.path.join(top_trans_path, state)
    for year in os.listdir(state_path):
        year_path = os.path.join(state_path, year)
        for file in os.listdir(year_path):
            file_path = os.path.join(year_path, file)
            with open(file_path, 'r') as f:
                data = json.load(f)
                if data.get("data"):

                    # States
                    states = data["data"].get("states", [])
                    if states:  # Check if not None
                        for entry in states:
                            top_transaction_data["State"].append(entry.get("entityName"))
                            top_transaction_data["Year"].append(int(year))
                            top_transaction_data["Quarter"].append(int(file.strip('.json')))
                            top_transaction_data["Pincode"].append(None)  # No Pincode for states
                            top_transaction_data["Transaction_count"].append(entry["metric"]["count"])
                            top_transaction_data["Transaction_amount"].append(entry["metric"]["amount"])

                    # Pincodes
                    pincodes = data["data"].get("pincodes", [])
                    if pincodes:
                        for entry in pincodes:
                            top_transaction_data["State"].append(state)
                            top_transaction_data["Year"].append(int(year))
                            top_transaction_data["Quarter"].append(int(file.strip('.json')))
                            top_transaction_data["Pincode"].append(entry.get("entityName"))
                            top_transaction_data["Transaction_count"].append(entry["metric"]["count"])
                            top_transaction_data["Transaction_amount"].append(entry["metric"]["amount"])

# Create DataFrame
df_top_transaction = pd.DataFrame(top_transaction_data)

print('top_transaction :')
print(df_top_transaction.head())

In [ ]:
#my ninth data frame.....top_insurance

import os
import json
import mysql.connector
import pandas as pd

# MySQL connection
conn = mysql.connector.connect(
    host="localhost",
    user="root",
    password="Amitpm@52",  # Update with your password
    database="phonepe_insights"
)
cursor = conn.cursor()

# Path for top_insurance JSON files
top_ins_path = "D:/guvi/project/phonepay/data/data/top/insurance/country/india/state"

# Define a dictionary to collect data
top_insurance_data = {
    'State': [], 'Year': [], 'Quarter': [],
    'Pincode': [], 'Insurance_count': [], 'Insurance_amount': []
}

# Loop through the files
for state in os.listdir(top_ins_path):
    state_path = os.path.join(top_ins_path, state)
    for year in os.listdir(state_path):
        year_path = os.path.join(state_path, year)
        for file in os.listdir(year_path):
            file_path = os.path.join(year_path, file)
            with open(file_path, 'r') as f:
                data = json.load(f)
                if data.get("data"):

                    # States
                    states = data["data"].get("states", [])
                    if states:  # Check if not None
                        for entry in states:
                            top_insurance_data["State"].append(entry.get("entityName"))
                            top_insurance_data["Year"].append(int(year))
                            top_insurance_data["Quarter"].append(int(file.strip('.json')))
                            top_insurance_data["Pincode"].append(None)  # No Pincode for states
                            top_insurance_data["Insurance_count"].append(entry["metric"]["count"])
                            top_insurance_data["Insurance_amount"].append(entry["metric"]["amount"])

                    # Pincodes
                    pincodes = data["data"].get("pincodes", [])
                    if pincodes:
                        for entry in pincodes:
                            top_insurance_data["State"].append(state)
                            top_insurance_data["Year"].append(int(year))
                            top_insurance_data["Quarter"].append(int(file.strip('.json')))
                            top_insurance_data["Pincode"].append(entry.get("entityName"))
                            top_insurance_data["Insurance_count"].append(entry["metric"]["count"])
                            top_insurance_data["Insurance_amount"].append(entry["metric"]["amount"])

# Create DataFrame
df_top_insurance = pd.DataFrame(top_insurance_data)
print("df_top_insurance:",df_top_insurance)

# Insert data into MySQL
for i, row in df_top_insurance.iterrows():
    sql = """
        INSERT INTO top_insurance (State, Year, Quarter, Pincode, Insurance_count, Insurance_amount)
        VALUES (%s, %s, %s, %s, %s, %s)
    """
    cursor.execute(sql, (row["State"], row["Year"], row["Quarter"], row["Pincode"], row["Insurance_count"], row["Insurance_amount"]))

# Commit and close connection
conn.commit()
cursor.close()
conn.close()

print("Data inserted into top_insurance successfully!")



In [ ]:
import mysql.connector
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# MySQL configuration
mysql_config = {
    'host': 'localhost',
    'user': 'root',
    'password': 'Amitpm@52',  # <-- replace with your password
    'database': 'phonepe_insights'
}

# Step 1: Connect and run the query
try:
    conn = mysql.connector.connect(**mysql_config)
    query = """
        SELECT 
            State, Year, SUM(Transaction_amount) AS Total_Transaction_Amount
        FROM 
            aggregated_transactions
        GROUP BY 
            State, Year
        ORDER BY 
            Total_Transaction_Amount DESC;
    """
    df = pd.read_sql(query, conn)
    print("Data fetched successfully.")
except mysql.connector.Error as err:
    print("MySQL Error:", err)
finally:
    if conn.is_connected():
        conn.close()

# Step 2: Print sample data
print(df.head())

# Step 3: Barplot – Top 10 State-Year combinations by total transaction amount
plt.figure(figsize=(14, 8))
top_10 = df.sort_values(by="Total_Transaction_Amount", ascending=False).head(10)
sns.barplot(data=top_10, x="Total_Transaction_Amount", y="State", hue="Year")
plt.title("Top 10 State-Year Pairs by Transaction Amount")
plt.xlabel("Total Transaction Amount")
plt.ylabel("State")
plt.tight_layout()
plt.show()

In [ ]:
# Step 4: Heatmap – State vs Year
pivot_table = df.pivot(index="State", columns="Year", values="Total_Transaction_Amount")
plt.figure(figsize=(14, 10))
sns.heatmap(pivot_table, annot=True, fmt=".1f", cmap="coolwarm")
plt.title("Transaction Volume Heatmap (State vs Year)")
plt.xlabel("Year")
plt.ylabel("State")
plt.tight_layout()
plt.show()


In [ ]:
!pip install cryptography


In [ ]:
import cryptography
print(cryptography.__version__)


In [ ]:
# MySQL configuration
mysql_config = {
    'host': 'localhost',
    'user': 'root',
    'password': 'Amitpm@52',  # <-- replace with your password
    'database': 'phonepe_insights'
}

# Step 1: Connect and run the query
try:
    conn = mysql.connector.connect(**mysql_config)
    query = """
SELECT 
    State, Year, SUM(Insurance_amount) AS Total_Insurance_Amount
FROM 
    map_insurance
GROUP BY 
    State, Year
ORDER BY 
    Total_Insurance_Amount DESC;
"""
    df = pd.read_sql(query, conn)
    print("Data fetched successfully.")
except mysql.connector.Error as err:
    print("MySQL Error:", err)
finally:
    if conn.is_connected():
        conn.close()

# Step 2: Print sample data
print(df.head())

In [ ]:
# Filter for most recent year
latest_year = df['Year'].max()
df_latest = df[df['Year'] == latest_year].sort_values('Total_Insurance_Amount', ascending=False)

plt.figure(figsize=(14, 6))
sns.barplot(data=df_latest, x='State', y='Total_Insurance_Amount', palette='viridis')
plt.xticks(rotation=90)
plt.title(f'Insurance Amount by State in {latest_year}')
plt.xlabel('State')
plt.ylabel('Total Insurance Amount (INR)')
plt.tight_layout()
plt.show()


In [ ]:
top5 = df_latest.head(5)

plt.figure(figsize=(5, 5))
plt.pie(top5['Total_Insurance_Amount'], labels=top5['State'], autopct='%1.1f%%', startangle=140)
plt.title(f'Top 5 States by Insurance Amount in {latest_year}')
plt.axis('equal')
plt.show()


In [ ]:
#4 Transaction Analysis for Market Expansion

import mysql.connector
import pandas as pd

# MySQL configuration
mysql_config = {
    'host': 'localhost',
    'user': 'root',
    'password': 'Amitpm@52',  # <-- use securely in real applications
    'database': 'phonepe_insights'
}

# Step 1: Connect and run the query
try:
    conn = mysql.connector.connect(**mysql_config)
    
    query = """
    SELECT 
        State, 
        SUM(Transaction_amount) AS Total_Transaction
    FROM 
        aggregated_transactions
    GROUP BY 
        State
    ORDER BY 
        Total_Transaction DESC;
    """
    
    df = pd.read_sql(query, conn)
    print("Data fetched successfully.")
    
except mysql.connector.Error as err:
    print("MySQL Error:", err)

finally:
    if conn.is_connected():
        conn.close()

# Step 2: Print sample data
print(df.head())


In [ ]:
# Bar Chart
plt.figure(figsize=(12, 6))
plt.bar(df['State'], df['Total_Transaction'], color='skyblue')
plt.xticks(rotation=90)
plt.title('Total Transactions by State')
plt.xlabel('State')
plt.ylabel('Total Transaction Amount')
plt.tight_layout()
plt.show()

In [ ]:
# Pie Chart - Top 10 States
top_states = df.head(10)
plt.figure(figsize=(5, 5))
plt.pie(top_states['Total_Transaction'], labels=top_states['State'], autopct='%1.1f%%', startangle=140)
plt.title('Top 10 States by Transaction Share')
plt.axis('equal')
plt.show()

In [ ]:
import mysql.connector
import pandas as pd
import matplotlib.pyplot as plt

# MySQL config
mysql_config = {
    'host': 'localhost',
    'user': 'root',
    'password': 'Amitpm@52',
    'database': 'phonepe_insights'
}

# Query and fetch
try:
    conn = mysql.connector.connect(**mysql_config)

    query = """
    SELECT
        State,
        SUM(RegisteredUsers) AS Total_RegisteredUsers
    FROM
        top_user
    GROUP BY
        State
    ORDER BY
        Total_RegisteredUsers DESC
    LIMIT 0, 1000;
    """

    df = pd.read_sql(query, conn)
    print("Data fetched successfully.")

except mysql.connector.Error as err:
    print("MySQL Error:", err)

finally:
    if conn.is_connected():
        conn.close()

# Sample data print
print(df.head())


In [ ]:
df.isnull().sum()

In [ ]:
df = df.dropna()


In [ ]:
df.head()

In [ ]:
# Bar Chart
plt.figure(figsize=(12, 6))
plt.bar(df['State'], df['Total_RegisteredUsers'], color='mediumseagreen')
plt.xticks(rotation=90)
plt.title('Total Registered Users by State')
plt.xlabel('State')
plt.ylabel('Total Registered Users')
plt.tight_layout()
plt.show()

In [ ]:
# Pie Chart – Top 10 States
top_states = df.head(10)
plt.figure(figsize=(6, 6))
plt.pie(top_states['Total_RegisteredUsers'], labels=top_states['State'], autopct='%1.1f%%', startangle=140)
plt.title('Top 10 States by Registered Users')
plt.axis('equal')
plt.show()

In [ ]:
mysql_config = {
    'host': 'localhost',
    'user': 'root',
    'password': 'Amitpm@52',
    'database': 'phonepe_insights'
}

# Step 1: Fetch data
try:
    conn = mysql.connector.connect(**mysql_config)
    query = """
    SELECT 
        State, 
        SUM(Insurance_amount) AS Total_Insurance
    FROM 
        top_insurance
    GROUP BY 
        State
    ORDER BY 
        Total_Insurance DESC;
    """
    df = pd.read_sql(query, conn)
    print("Data fetched successfully.")
except mysql.connector.Error as err:
    print("MySQL Error:", err)
finally:
    if conn.is_connected():
        conn.close()

# Step 2: Replace None with 0
df = df.fillna(0)

In [ ]:
df.head()

In [ ]:
# Step 3: Bar Chart
plt.figure(figsize=(12, 6))
plt.bar(df['State'], df['Total_Insurance'], color='skyblue')
plt.xticks(rotation=90)
plt.title("Total Insurance Amount by State")
plt.xlabel("State")
plt.ylabel("Total Insurance Amount")
plt.tight_layout()
plt.show()


In [ ]:
top10 = df.head(10)
plt.figure(figsize=(8, 8))
plt.pie(top10['Total_Insurance'], labels=top10['State'], autopct='%1.1f%%', startangle=140)
plt.title("Top 10 States by Total Insurance Amount")
plt.axis('equal')
plt.show()

In [ ]:
import mysql.connector
import pandas as pd
import matplotlib.pyplot as plt

# MySQL configuration
mysql_config = {
    'host': 'localhost',
    'user': 'root',
    'password': 'Amitpm@52',  # use your actual password
    'database': 'phonepe_insights'
}

# Step 1: Fetch data from database
try:
    conn = mysql.connector.connect(**mysql_config)
    query = """
    SELECT 
        State, Year, 
        SUM(RegisteredUsers) AS Total_Registered_Users, 
        SUM(AppOpens) AS Total_App_Opens
    FROM 
        aggregated_user
    GROUP BY 
        State, Year
    ORDER BY 
        Total_Registered_Users DESC;
    """
    df = pd.read_sql(query, conn)
    print("Data fetched successfully.")
except mysql.connector.Error as err:
    print("MySQL Error:", err)
finally:
    if conn.is_connected():
        conn.close()

# Step 2: Check and clean data
df = df.dropna()  # Remove rows with None or NaN values

# Optional: Group by state (all years combined)
grouped = df.groupby("State").agg({
    "Total_Registered_Users": "sum",
    "Total_App_Opens": "sum"
}).reset_index()

# Step 3: Bar Chart - Total App Opens by State
plt.figure(figsize=(12, 6))
plt.bar(grouped["State"], grouped["Total_App_Opens"], color="skyblue")
plt.xticks(rotation=90)
plt.title("Total App Opens by State")
plt.xlabel("State")
plt.ylabel("Total App Opens")
plt.tight_layout()
plt.show()



In [ ]:
import requests
import json

# Download GeoJSON data from GitHub
url = 'https://raw.githubusercontent.com/geohacker/india/master/state/india_states.geojson'
response = requests.get(url)

# Check status
if response.status_code == 200:
    data = response.json()  # Correct way to parse JSON
    with open("india_states.geojson", "w", encoding="utf-8") as f:
        json.dump(data, f)
    print("GeoJSON file downloaded and saved correctly!")
else:
    print("Failed to download. Status code:", response.status_code)


In [ ]:
import requests
import json

# Correct & Working GeoJSON URL
url = "https://raw.githubusercontent.com/datameet/maps/master/States/Admin2/india_states.geojson"

response = requests.get(url)

if response.status_code == 200:
    data = response.json()
    with open("india_states.geojson", "w", encoding="utf-8") as f:
        json.dump(data, f)
    print("GeoJSON file downloaded and saved successfully!")
else:
    print("Download failed with status code:", response.status_code)


In [ ]:
import json

file_path = r"C:\Users\USER\india_state_geo.json"

with open(file_path, "r", encoding="utf-8") as f:
    india_states = json.load(f)

print(india_states["features"][0]["properties"])


In [ ]:
import plotly.express as px
import pandas as pd

# Example DataFrame (aap apni dataframe use karna)
# State_clean column mein states ke naam honge jo GeoJSON mein properties ke saath match karte hain
state_amount = pd.DataFrame({
    "State_clean": ["Maharashtra", "Karnataka", "Tamil Nadu", "Delhi"],
    "Transaction_amount": [1000000, 750000, 500000, 300000]
})

# 'featureidkey' wo property hai jo GeoJSON file mein states ke naam ko hold karti hai
# Aap apne GeoJSON ko print karke check kar sakte ho (jaise: india_states['features'][0]['properties'])
# Example: properties mein 'ST_NM' ya 'NAME_1' ho sakta hai, aapke file pe depend karta hai

fig = px.choropleth(
    state_amount,
    geojson=india_states,
    locations="State_clean",
    color="Transaction_amount",
    featureidkey="properties.ST_NM",  # Yeh property aapko apne geojson se dekhni hai
    color_continuous_scale="Viridis",
    title="Transaction Amount by State in India"
)

fig.update_geos(fitbounds="locations", visible=False)
fig.update_layout(margin={"r":0,"t":30,"l":0,"b":0})

fig.show()
print(india_states['features'][0]['properties'].keys())



In [ ]:
import plotly.express as px
import pandas as pd

# Sample DataFrame (apne data se replace karna)
state_amount = pd.DataFrame({
    "State_clean": ["Maharashtra", "Karnataka", "Tamil Nadu", "Delhi"],
    "Transaction_amount": [1000000, 750000, 500000, 300000]
})

fig = px.choropleth(
    state_amount,
    geojson=india_states,
    locations="State_clean",       # DataFrame mein jo state name column hai
    color="Transaction_amount",    # DataFrame ka color column
    featureidkey="properties.NAME_1",  # GeoJSON mein state name key
    color_continuous_scale="Viridis",
    title="Transaction Amount by State in India"
)

fig.update_geos(fitbounds="locations", visible=False)
fig.update_layout(margin={"r":0,"t":30,"l":0,"b":0})

fig.show()


In [ ]:
!pip install shapely


In [ ]:
from shapely.geometry import shape

# Create dictionary: state name -> (longitude, latitude) of centroid
state_centroids = {}
for feature in india_states["features"]:
    state_name = feature["properties"]["NAME_1"]
    geometry = shape(feature["geometry"])
    centroid = geometry.centroid
    state_centroids[state_name] = (centroid.x, centroid.y)


In [ ]:
import pandas as pd

state_amount = pd.DataFrame({
    "State_clean": ["Maharashtra", "Karnataka", "Tamil Nadu", "Delhi"],
    "Transaction_amount": [1000000, 750000, 500000, 300000]
})


In [ ]:
import plotly.graph_objects as go

fig = go.Figure()

# Choropleth map with data
fig.add_trace(go.Choropleth(
    geojson=india_states,
    locations=state_amount["State_clean"],
    z=state_amount["Transaction_amount"],
    featureidkey="properties.NAME_1",
    colorscale="Viridis",
    colorbar_title="Transaction Amount",
    marker_line_color='black',
    marker_line_width=0.5
))

# Boundary outlines for all states (even those without data)
for feature in india_states["features"]:
    geometry = feature["geometry"]
    state_name = feature["properties"]["NAME_1"]

    if geometry["type"] == "Polygon":
        coords_list = [geometry["coordinates"]]
    elif geometry["type"] == "MultiPolygon":
        coords_list = geometry["coordinates"]
    else:
        continue

    for coords in coords_list:
        lon_vals = [pt[0] for pt in coords[0]]
        lat_vals = [pt[1] for pt in coords[0]]

        fig.add_trace(go.Scattergeo(
            lon=lon_vals,
            lat=lat_vals,
            mode='lines',
            line=dict(width=0.6, color='black'),
            hoverinfo='skip',
            showlegend=False
        ))

# State names
for state, (lon, lat) in state_centroids.items():
    fig.add_trace(go.Scattergeo(
        lon=[lon],
        lat=[lat],
        text=state,
        mode='text',
        showlegend=False,
        textfont=dict(color='black', size=9),
    ))

# Zoom into India
fig.update_geos(
    visible=False,
    projection_type="mercator",
    lataxis_range=[6, 38],
    lonaxis_range=[68, 98],
)

fig.update_layout(
    title_text="India Map with State-wise Transaction Amount and Boundaries",
    margin={"r":0,"t":30,"l":0,"b":0},
    height=650
)

fig.show()


In [ ]:
#insurance 

In [ ]:
import pandas as pd

# Example structure: Update as per your actual extracted data
insurance_df = pd.DataFrame({
    'State': ['Maharashtra', 'Karnataka', 'Delhi', 'Gujarat', 'Tamil Nadu'],
    'Year': [2022, 2022, 2022, 2022, 2022],
    'Quarter': [1, 1, 1, 1, 1],
    'Insurance_Transactions': [15000, 12000, 9000, 7000, 5000]
})


In [ ]:
import json

geo_path = r"C:\Users\USER\india_state_geo.json"

with open(geo_path, "r", encoding="utf-8") as f:
    india_geojson = json.load(f)


In [ ]:
import plotly.express as px

fig = px.choropleth(
    insurance_df,
    geojson=india_geojson,
    featureidkey='properties.NAME_1',  # Match state name field in geojson
    locations='State',
    color='Insurance_Transactions',
    color_continuous_scale='Purples',
    title='Top States by Insurance Transactions (PhonePe)',
    labels={'Insurance_Transactions': 'Transactions'},
    hover_name='State'
)

fig.update_geos(fitbounds="locations", visible=False)
fig.update_layout(margin={"r":0,"t":50,"l":0,"b":0})
fig.show()
